> Projeto Desenvolve <br>
Programação Intermediária com Python <br>
Profa. Camila Laranjeira (mila@projetodesenvolve.com.br) <br>

# 3.14 - ORM

## Exercícios

#### Q1. Conhecendo os dados
Baixe o seguinte csv onde iremos trabalhar. Ele contém informações sobre salários de profissionais de dados de uma empresa hipotética entre 2009 e 2016
* https://github.com/camilalaranjeira/python-intermediario/blob/main/salaries.csv

Suas colunas, descritas na [página do Kaggle que contém o dataset](https://www.kaggle.com/datasets/krishujeniya/salary-prediction-of-data-professions?resource=download), são:
* FIRST NAME: Primeiro nome do profissional de dados (String)
* LAST NAME: Sobrenome do profissional de dados (String)
* SEX: Gênero do profissional de dados (String: 'F' para Feminino, 'M' para Masculino)
* DOJ (Date of Joining): A data em que o profissional de dados ingressou na empresa (Data no formato MM/DD/AAAA)
* CURRENT DATE: A data atual ou a data de referência dos dados (Data no formato MM/DD/AAAA)
* DESIGNATION: O cargo ou designação do profissional de dados (String: ex., Analista, Analista Sênior, Gerente)
* AGE: Idade do profissional de dados (Integer)
* SALARY: Salário anual do profissional de dados (Float)
* UNIT: Unidade de negócios ou departamento em que o profissional de dados trabalha (String: ex., TI, Finanças, Marketing)
* LEAVES USED: Número de licenças utilizadas pelo profissional de dados (Integer)
* LEAVES REMAINING: Número de licenças restantes para o profissional de dados (Integer)
* RATINGS: Avaliações de desempenho do profissional de dados (Float)
* PAST EXP: Experiência de trabalho anterior em anos antes de ingressar na empresa atual (Float)

Na célula a seguir, **carregue os dados do CSV e dê uma olhada neles antes de seguir**.

In [1]:
import pandas as pd

# Carregar o arquivo CSV
df = pd.read_csv("salaries.csv")

# Mostrar as primeiras linhas
df.head()

,FIRST NAME,LAST NAME,SEX,DOJ,CURRENT DATE,DESIGNATION,AGE,SALARY,UNIT,LEAVES USED,LEAVES REMAINING,RATINGS,PAST EXP
0,TOMASA,ARMEN,F,5-18-2014,01-07-2016,Analyst,21.0,44570,Finance,24.0,6.0,2.0,0
1,ANNIE,NaN,F,NaN,01-07-2016,Associate,NaN,89207,Web,NaN,13.0,NaN,7
2,OLIVE,ANCY,F,7-28-2014,01-07-2016,Analyst,21.0,40955,Finance,23.0,7.0,3.0,0
3,CHERRY,AQUILAR,F,04-03-2013,01-07-2016,Analyst,22.0,45550,IT,22.0,8.0,3.0,0
4,LEON,ABOULAHOUD,M,11-20-2014,01-07-2016,Analyst,NaN,43161,Operations,27.0,3.0,NaN,3


#### Q2. Modelando os dados
Você deve **criar um ORM com SQLAlchemy capaz de comportar os dados dessa base**.

* Crie um campo de chave primária `ID`, que deve ser incrementado automaticamente
* Os campos SEX, DESIGNATION e UNIT devem ser definidos como classes `Enum` com os possíveis valores (consulte os valores únicos dessas colunas)
* Para os outros campos, consulte os tipos de dados informados na descrição acima

In [13]:
from sqlalchemy import create_engine, Column, Integer, String, Float, Date, Enum
from sqlalchemy.orm import declarative_base
from enum import Enum as PyEnum


# Criando a base do ORM
Base = declarative_base()


# Criando os Enums

class SexEnum(PyEnum):
    F = "F"
    M = "M"


class DesignationEnum(PyEnum):
    Analyst = "Analyst"
    Associate = "Associate"
    Senior_Analyst = "Senior Analyst"
    Senior_Manager = "Senior Manager"
    Manager = "Manager"
    Director = "Director"


class UnitEnum(PyEnum):
    Finance = "Finance"
    Web = "Web"
    IT = "IT"
    Operations = "Operations"
    Marketing = "Marketing"
    Management = "Management"


# Criando o modelo ORM

class Salary(Base):
    __tablename__ = "salaries"

    id = Column(Integer, primary_key=True, autoincrement=True)

    first_name = Column(String)
    last_name = Column(String)

    sex = Column(Enum(SexEnum))

    doj = Column(Date)
    current_date = Column(Date)

    designation = Column(Enum(DesignationEnum))

    age = Column(Integer)

    salary = Column(Float)

    unit = Column(Enum(UnitEnum))

    leaves_used = Column(Integer)
    leaves_remaining = Column(Integer)

    ratings = Column(Float)

    past_exp = Column(Float)


# Criando o banco SQLite e a tabela

engine = create_engine("sqlite:///salaries.db")

Base.metadata.create_all(engine)

print("Modelo ORM criado com sucesso!")

Modelo ORM criado com sucesso!


#### Q3. Estabelecendo uma conexão

Usando o método `create_engine` do SQLAlchemy, crie uma conexão com um novo banco de dados SQLite chamado `salarios`.

In [14]:
from sqlalchemy import create_engine

engine = create_engine("sqlite:///salarios.db")

print("Conexão criada com sucesso!")

Conexão criada com sucesso!


#### Q4. Criando as tabelas
Crie as tabelas da questão Q2 no banco `salarios`.

In [15]:
# Criando as tabelas no banco salarios

Base.metadata.create_all(engine)

print("Tabelas criadas com sucesso!")

Tabelas criadas com sucesso!


#### Q5. Populando

Usando o método `to_sql` da biblioteca Pandas (veja [a documentação](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_sql.html)), popule o banco `salarios` com os dados do csv que você carregou na questão Q1.
* Lembre-se de definir o parâmetro `if_exists='append'` para que as tabelas não sejam dropadas e recriadas.

In [12]:
# Renomeando as colunas do CSV para bater com o modelo ORM

df = df.rename(columns={
    "FIRST NAME": "first_name",
    "LAST NAME": "last_name",
    "SEX": "sex",
    "DOJ": "doj",
    "CURRENT DATE": "current_date",
    "DESIGNATION": "designation",
    "AGE": "age",
    "SALARY": "salary",
    "UNIT": "unit",
    "LEAVES USED": "leaves_used",
    "LEAVES REMAINING": "leaves_remaining",
    "RATINGS": "ratings",
    "PAST EXP": "past_exp"
})


# Inserindo no banco
df.to_sql(
    "salaries",
    con=engine,
    if_exists="append",
    index=False
)

print("Dados inseridos com sucesso!")

Dados inseridos com sucesso!


#### Q6. Consultas SQL vs ORM

Agrupe os dados por DESIGNATION e selecione o mínimo, máximo e a média dos salários (SALARY) divididos por 12. Já que o atributo SALARY é anual, dividir por 12 nos mostrará os valores mensais.

Assumindo que a variável que armazena a sua conexão se chama `engine`, você deve realizar a query acima de três formas:
* Executando a query SQL através de uma instância de conexão retornada pelo método `engine.connect()`
* Executando a query SQL com o método `read_sql_query` do Pandas (veja [a documentação](https://pandas.pydata.org/docs/reference/api/pandas.read_sql_query.html)). Você usará mesma instância `engine.connect()` como um dos parâmetros.
* Executando uma query criada com o módulo `select` do SQLAlchemy. Sua execução deve ser feita através de um objeto `Session` do módulo `orm` do SQLAlchemy (`Session(engine)`).


In [19]:
### Execute aqui sua query SQL com SQLAlchemy

from sqlalchemy import text

with engine.connect() as conn:
    resultado = conn.execute(text("""
        SELECT 
            designation,
            MIN(salary / 12) AS salario_minimo,
            MAX(salary / 12) AS salario_maximo,
            AVG(salary / 12) AS salario_medio
        FROM salaries
        GROUP BY designation
    """))

    for linha in resultado:
        print(linha)

('Analyst', 3333.4166666666665, 4165.0, 3751.675987685993)
('Associate', 5846.166666666667, 8300.25, 7266.915094339623)
('Director', 17832.25, 32342.666666666668, 23914.265625)
('Manager', 8343.666666666666, 12407.5, 10522.716049382716)
('Senior Analyst', 4170.333333333333, 5830.5, 4991.778792134832)
('Senior Manager', 12614.416666666666, 16631.416666666668, 14888.689516129032)


In [18]:
### Execute aqui sua query SQL com SQLAlchemy + Pandas

import pandas as pd

with engine.connect() as conn:
    resultado_df = pd.read_sql_query("""
        SELECT 
            designation,
            MIN(salary / 12) AS salario_minimo,
            MAX(salary / 12) AS salario_maximo,
            AVG(salary / 12) AS salario_medio
        FROM salaries
        GROUP BY designation
    """, conn)

resultado_df

,designation,salario_minimo,salario_maximo,salario_medio
0,Analyst,3333.416667,4165.000000,3751.675988
1,Associate,5846.166667,8300.250000,7266.915094
2,Director,17832.250000,32342.666667,23914.265625
3,Manager,8343.666667,12407.500000,10522.716049
4,Senior Analyst,4170.333333,5830.500000,4991.778792
5,Senior Manager,12614.416667,16631.416667,14888.689516


In [23]:
### Execute aqui sua query com SQLAlchemy ORM

from sqlalchemy.orm import Session
from sqlalchemy import select, func


with Session(engine) as session:

    consulta = select(
        Salary.designation.cast(String).label("designation"),
        func.min(Salary.salary / 12).label("salario_minimo"),
        func.max(Salary.salary / 12).label("salario_maximo"),
        func.avg(Salary.salary / 12).label("salario_medio")
    ).group_by(
        Salary.designation
    )

    resultado = session.execute(consulta)

    for linha in resultado:
        print(linha)

('Analyst', 3333.4166666666665, 4165.0, 3751.675987685993)
('Associate', 5846.166666666667, 8300.25, 7266.915094339623)
('Director', 17832.25, 32342.666666666668, 23914.265625)
('Manager', 8343.666666666666, 12407.5, 10522.716049382716)
('Senior Analyst', 4170.333333333333, 5830.5, 4991.778792134832)
('Senior Manager', 12614.416666666666, 16631.416666666668, 14888.689516129032)
